Vamos agora ajustar os parâmetros do FrodoKEM por forma a melhor perceber o que está em jogo na sua escolha. Considere-se uma versão reduzida caracterizada pelos seguintes parâmetros:

n
=
64
 -- dimensão da matriz
q
=
2
1
0
 -- módulo do anel dos elementos da matriz (
R
=
Z
q
)
n
¯
=
m
¯
=
8
 -- dimensão dos blocos do segredo/ruído
B
=
2
 -- bits da mensagem codificados em cada entrada da matriz
χ
 -- distribuição ajustada para amostragem dos segredos/ruído (
χ
≈
1.0
)
A tabela de probabilidades acumuladas (com 14 bits de precisão):

In [1]:
n=64
q=2^(10)
n_=m=8
B=2

In [2]:
def genMat(seedA,q,n):
    R = IntegerModRing(q)
    set_random_seed(seedA)
    A = random_matrix(R,n,n)
    return A

In [3]:
T_CHI64 = [6536, 14465, 16234, 16379, 16384]
def chiSample(linhas,colunas,q,tabela = T_CHI64):
    R = ZZ
    matriz = matrix(R,linhas,colunas)
    for l in range(linhas):
        for c in range(colunas):
            x = randint(0, 2^(14)-1)
            i = 0
            while i < len(tabela):
                if tabela[i] > x:
                    break
                i+=1

            bit_sinal = randint(0,1)
            if bit_sinal:
                s = -1
            else:
                s = 1;
            matriz[l, c] = s * i
    return matriz

In [4]:
def encode (bits, n, enq, B = 2): 
    assert len(bits) == n * n * B

    R = Integers(q)
    scale = q // (2^B)

    M = Matrix(R, n, n)

    idx = 0
    for i in range(n):
        for j in range(n):
            # pega B bits
            val = 0
            for b in range(B):
                val = (val << 1) | bits[idx]
                idx += 1

            M[i, j] = R(val * scale)

    return M

In [5]:
def decode (M, q , B = 2):
    n = M.nrows()
    bits = []

    for i in range(n):
        for j in range(n):
            x = Integer(M[i, j])

            val = round(x * (2^B) / q)

            val = val % (2^B)

            for b in reversed(range(B)):
                bits.append((val >> b) & 1)

    return bits

In [6]:
def key_Gen(q,table,n,n_,):
    R = IntegerModRing(q)
    seedA= randint(0,q-1)
    A=genMat(seedA,q,n)
    S = chiSample(n, n_, table)
    E = chiSample(n, n_, table)
    B_pk=A*S+E
    pk=(seedA,B_pk)
    sk=S
    return (sk,pk)

In [7]:
def enc(pk,m,table,n,n_,q):
    R = IntegerModRing(q)
    seedA,B_pk=pk
    A=genMat(seedA,q,n)
    S_=chiSample(n_,n,table)
    E_=chiSample(n_,n,table)
    E__=chiSample(n_,n_,table)
    c1=S_*A+E_
    V_=S_*B_pk+E__
    M = encode(m, n_, q)
    c2=V_+M
    return(c1,c2)

In [8]:
def dec(sk,C,pk,q,n):
    c1,c2=C
    S=sk
    seedA,B_pk=pk
    A=genMat(seedA,q,n)
    V= c1*S
    M=c2-V
    m_=decode(M,q)
    return m_

In [9]:
m_bits = [randint(0, 1) for _ in range(n_ * n_ * B)]

matriz_inicial=encode(m_bits, n_, q, B)
show(matriz_inicial)
sk, pk = key_Gen(q, T_CHI64, n, n_)
C = enc(pk, m_bits, T_CHI64, n, n_, q)
m_ = dec(sk, C, pk, q, n)


c1, c2 = C
V = c1 * sk
matriz_final = c2 - V
show(matriz_final)





[  0 512 512 512 256 768   0 512]
[  0 512 256   0   0 256 256   0]
[  0   0   0 512   0 512 256   0]
[768   0 256   0   0 512 768   0]
[512 256 768   0   0 512 256 768]
[512 512 768 256 512 768   0 512]
[  0 256 768 256 768 256 512 768]
[256 256 512   0 256 256 256 512]

[1020  496  526  523  257  748   19  508]
[1014  543  262    7 1016  243  233   17]
[   5    2   10  521 1003  512  261    0]
[ 750 1008  249   17 1005  518  772 1023]
[ 514  255  747 1015 1023  503  250  771]
[ 515  480  780  265  499  788    4  517]
[  14  242  770  260  758  260  530  753]
[ 253  263  521    6  265  248  262  519]

In [10]:
m_bits

[0,
 0,
 1,
 0,
 1,
 0,
 1,
 0,
 0,
 1,
 1,
 1,
 0,
 0,
 1,
 0,
 0,
 0,
 1,
 0,
 0,
 1,
 0,
 0,
 0,
 0,
 0,
 1,
 0,
 1,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 1,
 0,
 0,
 0,
 1,
 0,
 0,
 1,
 0,
 0,
 1,
 1,
 0,
 0,
 0,
 1,
 0,
 0,
 0,
 0,
 1,
 0,
 1,
 1,
 0,
 0,
 1,
 0,
 0,
 1,
 1,
 1,
 0,
 0,
 0,
 0,
 1,
 0,
 0,
 1,
 1,
 1,
 1,
 0,
 1,
 0,
 1,
 1,
 0,
 1,
 1,
 0,
 1,
 1,
 0,
 0,
 1,
 0,
 0,
 0,
 0,
 1,
 1,
 1,
 0,
 1,
 1,
 1,
 0,
 1,
 1,
 0,
 1,
 1,
 0,
 1,
 0,
 1,
 1,
 0,
 0,
 0,
 0,
 1,
 0,
 1,
 0,
 1,
 1,
 0]

In [11]:
m_

[0,
 0,
 1,
 0,
 1,
 0,
 1,
 0,
 0,
 1,
 1,
 1,
 0,
 0,
 1,
 0,
 0,
 0,
 1,
 0,
 0,
 1,
 0,
 0,
 0,
 0,
 0,
 1,
 0,
 1,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 1,
 0,
 0,
 0,
 1,
 0,
 0,
 1,
 0,
 0,
 1,
 1,
 0,
 0,
 0,
 1,
 0,
 0,
 0,
 0,
 1,
 0,
 1,
 1,
 0,
 0,
 1,
 0,
 0,
 1,
 1,
 1,
 0,
 0,
 0,
 0,
 1,
 0,
 0,
 1,
 1,
 1,
 1,
 0,
 1,
 0,
 1,
 1,
 0,
 1,
 1,
 0,
 1,
 1,
 0,
 0,
 1,
 0,
 0,
 0,
 0,
 1,
 1,
 1,
 0,
 1,
 1,
 1,
 0,
 1,
 1,
 0,
 1,
 1,
 0,
 1,
 0,
 1,
 1,
 0,
 0,
 0,
 0,
 1,
 0,
 1,
 0,
 1,
 1,
 0]

In [12]:
print(m_bits == m_)

count=0
for i in range(len(m_bits)):
    if m_bits[i] != m_[i]:
        count+=1


print(count)

True
0
